# Cookie Cats A/B Testi — 01: Deney Tanımı ve Veri Kontrolü

## Adım 1: Deney tanımı

**Bağlam:** Cookie Cats mobil bir bulmaca oyunu. Oyuncular belirli seviyelerde bir *kapıya* (gate) takılır; ya beklemek ya da uygulama içi satın alma yapmak zorunda kalır. Deneyde oyuncular kurulum anında rastgele iki gruba ayrılmış:

| Grup | Kapının yeri |
|---|---|
| `gate_30` (kontrol) | Level 30 |
| `gate_40` (test) | Level 40 |

**Problem:** Kapı level 30'dan 40'a taşınırsa oyuncuların oyunda kalma oranı değişir mi?

**Hipotezler** (her metrik için):
- H0: İki grup arasında retention farkı yoktur (p_30 = p_40)
- H1: Fark vardır (p_30 ≠ p_40), çift yönlü test

**Metrikler:**
| Rol | Metrik | Tanım |
|---|---|---|
| Ana metrik | `retention_7` | Oyuncu kurulumdan 7 gün sonra oyuna döndü mü |
| Yardımcı | `retention_1` | Oyuncu kurulumdan 1 gün sonra döndü mü |
| Koruma (guardrail) | `sum_gamerounds` | İlk 14 günde oynanan toplam round |

**Anlamlılık düzeyi:** α = 0.05

**Neden ana metrik retention_7?** Kapının etkisi ancak oyuncu kapıya ulaştıktan sonra ortaya çıkabilir. Verideki medyan oyuncu 14 günde yalnızca 16 round oynuyor ve en fazla %37'si 30 round'a ulaşıyor (notebook 02). 1. günde kapıyı gören oyuncu az olduğu için etki retention_1'de sulanır; retention_7 ise kapıyla karşılaşmış oyuncuların geri dönüp dönmediğini yakalar.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare

pd.set_option("display.float_format", "{:.4f}".format)
df = pd.read_csv("../data/cookie_cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## Adım 2: Veri kontrolü

### 2.1 Boyut, tipler, eksik değer, tekrar eden oyuncu

In [2]:
print("Boyut:", df.shape)
print(df.dtypes, "\n")
print("Eksik değer sayısı:", df.isna().sum().sum())
print("Tekrar eden userid:", df["userid"].duplicated().sum())

Boyut: (90189, 5)
userid            int64
version             str
sum_gamerounds    int64
retention_1        bool
retention_7        bool
dtype: object 

Eksik değer sayısı: 0
Tekrar eden userid: 0


### 2.2 Grup dengesi: Sample Ratio Mismatch (SRM) kontrolü

Atama 50/50 tasarlandıysa grup büyüklükleri de yaklaşık eşit olmalı. Ki-kare uyum iyiliği testiyle kontrol ediyoruz. SRM varsa (genelde p < 0.01 eşiği kullanılır) atamada ya da veri toplamada bir sorun olabilir ve sonuçlara güven azalır.

In [3]:
counts = df["version"].value_counts().sort_index()
share = counts / counts.sum()
display(pd.DataFrame({"oyuncu": counts, "pay": share}))

stat, p_srm = chisquare(counts.values)   # beklenen: eşit dağılım
print(f"Fark: {counts.max() - counts.min()} oyuncu")
print(f"SRM ki-kare = {stat:.3f}, p = {p_srm:.4f}")

,oyuncu,pay
version,,
gate_30,44700,0.4956
gate_40,45489,0.5044


Fark: 789 oyuncu
SRM ki-kare = 6.902, p = 0.0086


**Değerlendirme:** p ≈ 0.009 ile istatistiksel olarak anlamlı bir SRM var (gruplar %49.6 / %50.4). Olası nedenler: rastgeleleştirme hatası, bir grupta loglama kaybı ya da kurulumdan hemen sonra çıkan oyuncuların gruplarda farklı kaydedilmesi.

Atama mekanizması hakkında bilgi olmadığı için nedeni belirlenemiyor. Analize devam ediyorum, ancak sonuçlar bu **sınırlama** ile birlikte raporlanacak. Gerçek bir ortamda ilk adım, mühendislik ekibiyle atama ve loglama sistemini incelemek; neden bulunamazsa deneyi tekrarlamak olurdu.

### 2.3 `sum_gamerounds` dağılımı ve aşırı uçlar

In [4]:
df["sum_gamerounds"].describe(percentiles=[.5, .9, .99, .999])

count   90189.0000
mean       51.8725
std       195.0509
min         0.0000
50%        16.0000
90%       134.0000
99%       493.0000
99.9%    1073.6240
max     49854.0000
Name: sum_gamerounds, dtype: float64

In [5]:
df.nlargest(5, "sum_gamerounds")

,userid,version,sum_gamerounds,retention_1,retention_7
57702,6390605,gate_30,49854,False,True
7912,871500,gate_30,2961,True,True
29417,3271615,gate_40,2640,True,False
43671,4832608,gate_30,2438,True,True
48188,5346171,gate_40,2294,True,True


In [6]:
top, second = df["sum_gamerounds"].nlargest(2)
print(f"En yüksek: {top} round | ikinci: {second} round | oran: {top/second:.1f}x")
print(f"14 günde {top} round ≈ günde {top/14:.0f} round")

En yüksek: 49854 round | ikinci: 2961 round | oran: 16.8x
14 günde 49854 round ≈ günde 3561 round


**Karar:** 49.854 round, ikinci en yüksek değerin ~17 katı ve günde ~3.500 round demek; fiziksel olarak mümkün değil, veri hatası olarak kabul ediyorum. **Yalnızca bu tek kayıt** notebook 02'de çıkarılıyor.

Diğer yüksek değerler tutuluyor: `sum_gamerounds` deneyden etkilenen bir değişken, ona göre filtreleme yapmak gruplar arasında yanlılık yaratır (post-treatment bias) ve kapıya en çok maruz kalan oyuncuları siler. Round analizinde aykırı değerlere dayanıklı yöntemler (medyan, Mann-Whitney U) kullanılıyor.

### 2.4 Hiç oynamayan oyuncular

In [7]:
zero = df[df["sum_gamerounds"] == 0]
print(f"0 round oynayan: {len(zero)} oyuncu (%{100*len(zero)/len(df):.1f})")
display(zero["version"].value_counts())
print("Bunların retention_1 oranı:", round(zero["retention_1"].mean(), 4))

0 round oynayan: 3994 oyuncu (%4.4)


version
gate_40    2057
gate_30    1937
Name: count, dtype: int64

Bunların retention_1 oranı: 0.0218


**Karar:** Bu 3.994 oyuncu (%4.4) kapıya hiç ulaşmadığı için kapıdan etkilenemez, ama atama anında rastgele gruplara düştüler. **Intent-to-treat** yaklaşımıyla analizde tutuyorum: atanan herkesi analiz etmek standart ve tarafsız yaklaşımdır. Sayıları iki grupta benzer (1.937 / 2.057), bu yüzden sonucun yönünü değiştirmezler.

### 2.5 Mantık kontrolü: 7. gün dönen ama 1. gün dönmeyenler

In [8]:
odd = (df["retention_7"] & ~df["retention_1"]).sum()
print(f"retention_7=True ama retention_1=False: {odd} oyuncu (%{100*odd/len(df):.1f})")

retention_7=True ama retention_1=False: 3599 oyuncu (%4.0)


Bu bir hata değil: retention metrikleri "o gün döndü mü" anlamına geliyor, "o güne kadar hep oynadı mı" anlamına gelmiyor. 1. gün gelmeyip 7. gün dönen oyuncu olabilir. Metrik tanımını doğru anladığını göstermek için bu notu raporda tut.

### 2.6 İlk bakış: gruplara göre ham retention oranları (test öncesi)

In [9]:
df.groupby("version")[["retention_1", "retention_7"]].mean()

,retention_1,retention_7
version,,
gate_30,0.4482,0.1902
gate_40,0.4423,0.1820


## Adım 2 özeti

| Kontrol | Sonuç |
|---|---|
| Eksik değer | Yok |
| Tekrar eden oyuncu | Yok |
| Grup dengesi (SRM) | ⚠️ Anlamlı dengesizlik (p ≈ 0.009) |
| Aşırı uç | 1 oyuncu, 49.854 round |
| Hiç oynamayan | 3.994 oyuncu (~%4.4) |

**Veri kalitesi değerlendirmesi:** Veri temiz (eksik değer ve tekrar eden oyuncu yok), ancak iki konu dikkat gerektiriyor. Gruplar arasında anlamlı bir örneklem dengesizliği (SRM) var; sonuçlar bu sınırlamayla raporlanacak. Bir veri hatası (49.854 round) çıkarılacak, diğer yüksek değerler ve hiç oynamayan oyuncular tutulacak.